In [1]:
# NUM_CALIBRATION_SAMPLES 256 -> 128

# Import

In [2]:
import psutil
ram = psutil.virtual_memory()
print(f"전체 RAM: {ram.total / 1e9:.1f} GB")
print(f"사용 가능: {ram.available / 1e9:.1f} GB")

전체 RAM: 25.8 GB
사용 가능: 11.6 GB


In [3]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

/Users/jab/miniforge3/envs/ml/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/Users/jab/miniforge3/envs/ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# =============================================
# GPU 디바이스 자동 감지
# ─── Apple Silicon Mac → MPS (Metal GPU)
# ─── NVIDIA GPU       → CUDA
# ─── 없음             → CPU
#
# ⚠️ llmcompressor oneshot() 제약:
#   GPTQ/AWQ(oneshot)는 CUDA/XPU만 지원 → MPS Mac에서는 CPU 강제
#   (실행 시 "CUDA/XPU is not available! Compressing on CPU" 경고 정상)
#   현재 코드에서 DEVICE는 모델 로드/검증 등에만 사용 가능
# =============================================
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("[INFO] ✅ Apple MPS (GPU) 감지")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"[INFO] ✅ CUDA GPU 감지: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    print("[INFO] CPU 사용 (GPU 없음)")

print(f"[INFO] DEVICE = {DEVICE}")
print("[INFO] ⚠️  GPTQ(oneshot)는 llmcompressor 제약으로 CPU 전용 실행")

[INFO] ✅ Apple MPS (GPU) 감지
[INFO] DEVICE = mps
[INFO] ⚠️  GPTQ(oneshot)는 llmcompressor 제약으로 CPU 전용 실행


# Setting

In [5]:
MODEL_ID = "./base_model"     
OUT_DIR  = "./model"          

DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

NUM_CALIBRATION_SAMPLES = 128
MAX_SEQUENCE_LENGTH = 512

# Quantization
SCHEME = "W4A16"
TARGETS = ["Linear"]
IGNORE  = ["embed_tokens", "lm_head"]

# Model Loads

In [6]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
)

print("[INFO] 모델/토크나이저 로드 완료")

`torch_dtype` is deprecated! Use `dtype` instead!


[INFO] 모델 로드 중...
[INFO] 모델/토크나이저 로드 완료


# Dataset Loads & Preprocess

In [7]:
print("[INFO] 캘리브레이션 데이터 로드 중...")

ds = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False)
    }

ds = ds.map(preprocess)

print("[INFO] 데이터 전처리 완료")

[INFO] 캘리브레이션 데이터 로드 중...
[INFO] 데이터 전처리 완료


# GPTQ Quantization

In [ ]:
print(f"[INFO] GPTQ 시작 (scheme={SCHEME}, samples={NUM_CALIBRATION_SAMPLES}, max_len={MAX_SEQUENCE_LENGTH})...")

# ── GPU 가속: CUDA 자동 감지 ──
if torch.cuda.is_available():
    model.to("cuda")
    print(f"[INFO] 양자화: CUDA GPU ({torch.cuda.get_device_name(0)})")
else:
    model.to("cpu")
    print("[INFO] 양자화: CPU 모드 (MPS는 llmcompressor 미지원)")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("[INFO] GPTQ 완료")

# Model Save

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

print(f"[INFO] 모델 저장 완료: {OUT_DIR}")

2026-02-22T20:46:02.173080+0900 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 210it [00:01, 146.55it/s]


[INFO] 모델 저장 완료: ./model


# Submission

In [ ]:
zip_name = "Try_000"
print(f"[INFO] {zip_name}.zip 생성 중...")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

print(f"[INFO] 생성 완료: {zip_name}.zip")

[INFO] Try_000.zip 생성 중...
[INFO] 생성 완료: Try_000.zip
